# FASTA Predict

In [ ]:
%pip install -qq --ignore-requires-python --no-deps 'graphies[predict] @ git+https://github.com/lukasmki/graphies.git'
%pip install -qq pydantic networkx datasets polars torch

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print(
        "CUDA is not available. Please ensure you have selected a GPU runtime in 'Runtime > Change runtime type'."
    )

## Setup Data Sources

In [ ]:
!mkdir -p fasta-data
!wget -nv https://raw.githubusercontent.com/lukasmki/graphies-applications/refs/heads/main/grammars/fasta.json -O fasta-data/selfies.json

from pathlib import Path

GRAMMAR_PATH = next(Path().glob("fasta-data/*.json"))

In [ ]:
%pip install -qq selfies

from datasets import load_dataset

import selfies as sf

sf.set_semantic_constraints(
    {  # disable degree constriaints
        k: 24 for k, _ in sf.get_semantic_constraints().items()
    }
)


def transform_batch(batch: dict):
    fastas = []
    for seq in batch["sequence"]:
        fastas.append("".join([f"[{c}]" for c in seq.lower()]))

    return {"graphies": fastas}


hf_dataset = load_dataset("lukaskim/ChEMBL-36", "targets", split="train")

hf_dataset = hf_dataset.map(
    transform_batch,
    batched=True,
    batch_size=1000,
    num_proc=4,
    desc="Encoding FASTA",
)

hf_dataset = hf_dataset.filter(
    lambda x: x["graphies"] is not None,
    num_proc=4,
    desc="Filtering nulls",
)

## Setup Data Loaders

In [ ]:
from graphies.predict import GraphiesTokenizer, HFGraphiesDataset
from torch.utils.data import DataLoader, random_split

tokenizer = GraphiesTokenizer(GRAMMAR_PATH)
dataset = HFGraphiesDataset(
    hf_dataset, column="graphies", tokenizer=tokenizer, split=None
)

trn, tst = random_split(dataset, [0.9, 0.1])
torch.save(
    {"trn_indices": trn.indices, "tst_indices": tst.indices}, "fasta-data/split.pt"
)
trn_loader = DataLoader(
    dataset=trn,
    batch_size=256,
    shuffle=True,
    collate_fn=tokenizer.collate,
)
tst_loader = DataLoader(
    dataset=tst,
    batch_size=256,
    shuffle=False,
    collate_fn=tokenizer.collate,
)

## Trainer

In [ ]:
from graphies.predict import GraphiesTrainer
from graphies.predict.models import GRU
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRU(vocab_size=tokenizer.vocab_size)
optimizer = Adam(params=model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

# include kwargs for classes to restart from checkpoitn
checkpoint = {
    "model_kwargs": {"vocab_size": tokenizer.vocab_size},
    "optimizer_kwargs": {"lr": 1e-3},
    "scheduler_kwargs": {"mode": "min", "patience": 3, "factor": 0.1},
}
trainer = GraphiesTrainer(model, optimizer, scheduler, device, checkpoint)

In [ ]:
trainer.train(
    train=trn_loader,
    epochs=1,
    test=tst_loader,
    test_interval=1,
    log="fasta-data/log.csv",
    log_interval=1,
    checkpoint="fasta-data/chk.pt",
    checkpoint_interval=1,
)
trainer.save_model("fasta-data/model.pt")

In [ ]:
# export to google drive
from google.colab import drive

drive.mount("/content/drive")
!zip -r fasta-data.zip fasta-data/
!cp fasta-data.zip '/content/drive/MyDrive/'

# Run Inference

In [ ]:
%pip install -qq pymsaviz

import os
import tempfile

from graphies.predict import GraphiesModel
from IPython.display import Image, display
from pymsaviz import MsaViz

sf.set_semantic_constraints("default")

model = GraphiesModel.from_checkpoint(
    checkpoint="fasta-data/model.pt",
    tokenizer=tokenizer,
    model_cls=GRU,
    device=device,
)

fastas = []
sequences = model.generate(num=32, temperature=0.9, top_p=0.95, max_len=200)
for i, sequence in enumerate(sequences):
    graphies = tokenizer.strip(sequence)  # strip [BEGIN] and [END] tags
    fasta = graphies.replace("[", "").replace("]", "").upper()
    fastas.append((f"seq_{i:03d}", fasta))

with tempfile.NamedTemporaryFile(mode="w", suffix=".fa", delete=False) as tmp:
    tmp_path = tmp.name
    for seq_id, seq in fastas:
        tmp.write(f">{seq_id}\n{seq}\n")

mv = MsaViz(
    tmp_path,
    color_scheme="Zappo",
    wrap_length=60,
    show_consensus=True,
    show_count=True,
)

mv.savefig("sequences.png", dpi=150)
display(Image("sequences.png"))

os.unlink(tmp_path)